# Initial Setup (+ Huggingface Token)

In [20]:
!pip install -qU torch transformers datasets peft trl accelerate bitsandbytes

In [2]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)

In [36]:
import warnings
warnings.filterwarnings("ignore")

# 1. Load data & tokenizer

In [24]:
import json
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

dataset = load_dataset("Salesforce/xlam-function-calling-60k", split="train")
dataset = dataset.select(range(1600)).train_test_split(test_size=100, seed=42)
train_dataset = dataset["train"]
test_dataset  = dataset["test"]  # ✅ kept for benchmarking

In [25]:
def format_tool_calling(row):
    system_prompt = (
        "You are a helpful assistant with access to the following functions. "
        f"Use them if required:\n{row['tools']}\n\n"
        "Respond strictly with a JSON array of function calls."
    )
    messages = [
        {"role": "system",    "content": system_prompt},
        {"role": "user",      "content": row["query"]},
        {"role": "assistant", "content": row["answers"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

train_dataset = train_dataset.map(format_tool_calling, remove_columns=train_dataset.column_names)
print("Train size:", len(train_dataset), "| Test size:", len(test_dataset))

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Train size: 1500 | Test size: 100


# 2. Inference helper & benchmark function

In [30]:
from tqdm import tqdm

def run_inference(model_, query: str, tools_str: str) -> str:
    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant with access to the following functions. "
            f"Use them if required:\n{tools_str}\n\n"
            "Respond strictly with a JSON array of function calls."
        )},
        {"role": "user", "content": query},
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model_.device)
    with torch.no_grad():
        outputs = model_.generate(**inputs, max_new_tokens=256, do_sample=False)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)


def safe_parse_json(value, fallback=None):
    """Safely parse a value that may already be a dict/list or a JSON string."""
    if value is None:
        return fallback
    if isinstance(value, (dict, list)):
        return value
    if isinstance(value, str):
        try:
            return json.loads(value)
        except (json.JSONDecodeError, ValueError):
            return fallback
    return fallback


def safe_get_args(call: dict) -> dict:
    """Extract arguments from a function call dict, handling all known key variants."""
    for key in ("arguments", "parameters", "args", "input"):
        if key in call:
            result = safe_parse_json(call[key], fallback={})
            if isinstance(result, dict):
                return result
    return {}


def benchmark(model_, test_ds, label="Model") -> dict:
    json_valid = name_match = args_exact = args_keys_match = 0
    n = len(test_ds)

    for row in tqdm(test_ds, desc=f"Benchmarking [{label}]", unit="sample"):

        # --- guard: skip rows with missing fields ---
        if not isinstance(row.get("query"), str) or not isinstance(row.get("tools"), str):
            n -= 1
            continue

        # --- inference ---
        try:
            pred_str = run_inference(model_, row["query"], row["tools"])
        except Exception:
            continue

        # --- parse prediction ---
        pred = safe_parse_json(pred_str, fallback=None)
        if not isinstance(pred, list):
            continue
        json_valid += 1

        # --- parse ground truth ---
        gold = safe_parse_json(row.get("answers"), fallback=None)
        if not isinstance(gold, list):
            n -= 1
            continue

        gold_calls = [c for c in gold if isinstance(c, dict)]
        pred_calls = [c for c in pred if isinstance(c, dict)]

        if not gold_calls:
            n -= 1
            continue

        # --- name match ---
        gold_names = {c.get("name") for c in gold_calls if c.get("name")}
        pred_names = {c.get("name") for c in pred_calls if c.get("name")}
        if gold_names and gold_names == pred_names:
            name_match += 1

        # --- argument precision (first call) ---
        g_args = safe_get_args(gold_calls[0])
        p_args = safe_get_args(pred_calls[0]) if pred_calls else {}

        if g_args.keys() == p_args.keys():
            args_keys_match += 1
        if g_args == p_args:
            args_exact += 1

    if n == 0:
        print(f"[{label}] No valid rows to evaluate.")
        return {}

    results = {
        "label":             label,
        "total":             n,
        "json_valid %":      round(100 * json_valid       / n, 1),
        "name_match %":      round(100 * name_match       / n, 1),
        "args_keys_match %": round(100 * args_keys_match  / n, 1),
        "args_exact %":      round(100 * args_exact       / n, 1),
    }

    print(f"\n{'='*45}")
    print(f"  Benchmark: {label}")
    print(f"{'='*45}")
    for k, v in results.items():
        print(f"  {k:<22}: {v}")
    print(f"{'='*45}\n")
    return results

# 3. Load base model & run PRE benchmark

In [31]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
)
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

# ✅ Benchmark BEFORE fine-tuning
results_before = benchmark(model, test_dataset, label="Base model (before fine-tuning)")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Memory footprint: 0.99 GB


Benchmarking [Base model (before fine-tuning)]: 100%|██████████| 100/100 [03:27<00:00,  2.08s/sample]


  Benchmark: Base model (before fine-tuning)
  label                 : Base model (before fine-tuning)
  total                 : 100
  json_valid %          : 41.0
  name_match %          : 24.0
  args_keys_match %     : 14.0
  args_exact %          : 10.0



# 4.  Fine-tuning (LoRA)

In [32]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# Required for PEFT + SFTTrainer
model.config.use_cache = False
model.enable_input_require_grads()

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

training_args = SFTConfig(
    output_dir="./qwen_function_calling",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    logging_steps=10,
    max_steps=150,
    optim="adamw_torch",
    fp16=True,
    save_strategy="no",
    dataset_text_field="text",
    max_length=1024,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

print("Starting fine-tuning...")
trainer.train()

Adding EOS to train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting fine-tuning...


Step,Training Loss
10,1.163874
20,0.835123
30,0.883111
40,0.767119
50,0.837291
60,0.807282
70,0.739387
80,0.751575
90,0.768347
100,0.815859


TrainOutput(global_step=150, training_loss=0.8122496096293131, metrics={'train_runtime': 267.6735, 'train_samples_per_second': 4.483, 'train_steps_per_second': 0.56, 'total_flos': 1764482271129600.0, 'train_loss': 0.8122496096293131})

# 5. Save, load & run POST benchmark

In [33]:
from peft import PeftModel

# Save adapter
trainer.model.save_pretrained("./qwen_fc_lora")
tokenizer.save_pretrained("./qwen_fc_lora")
print("Adapter saved.")

# Load fine-tuned model
base  = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")
merged = PeftModel.from_pretrained(base, "./qwen_fc_lora")

# ✅ Benchmark AFTER fine-tuning
results_after = benchmark(merged, test_dataset, label="Fine-tuned model (after LoRA)")

Adapter saved.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Benchmarking [Fine-tuned model (after LoRA)]: 100%|██████████| 100/100 [04:19<00:00,  2.60s/sample]


  Benchmark: Fine-tuned model (after LoRA)
  label                 : Fine-tuned model (after LoRA)
  total                 : 100
  json_valid %          : 96.0
  name_match %          : 94.0
  args_keys_match %     : 79.0
  args_exact %          : 69.0



# 6. Side-by-side comparison & ask_function

In [34]:
# --- Summary table ---
print(f"\n{'Metric':<25} {'Before':>10} {'After':>10} {'Delta':>10}")
print("-" * 57)
metrics = ["json_valid %", "name_match %", "args_keys_match %", "args_exact %"]
for m in metrics:
    b = results_before[m]
    a = results_after[m]
    delta = f"{a - b:+.1f}"
    print(f"{m:<25} {b:>10} {a:>10} {delta:>10}")


Metric                        Before      After      Delta
---------------------------------------------------------
json_valid %                    41.0       96.0      +55.0
name_match %                    24.0       94.0      +70.0
args_keys_match %               14.0       79.0      +65.0
args_exact %                    10.0       69.0      +59.0


In [38]:
# --- ask_function utility ---
def ask_function(user_query: str, tools: list) -> list:
    tools_str = json.dumps(tools)
    response = run_inference(merged, user_query, tools_str)
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        return [{"error": "invalid JSON", "raw": response}]


# --- Example calls ---
tools = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a location",
        "parameters": {
            "location": {"type": "string", "description": "City name"},
            "unit":     {"type": "string", "enum": ["celsius", "fahrenheit"]}
        }
    }
]

print(json.dumps(ask_function("What's the weather in Paris?", tools), indent=2))
print(json.dumps(ask_function("My hometown is Astana. What's the weather in my hometown?", tools), indent=2))

[
  {
    "name": "get_weather",
    "arguments": {
      "location": "Paris"
    }
  }
]
[
  {
    "name": "get_weather",
    "arguments": {
      "location": "Astana"
    }
  }
]


# 7. Qwen Merged GGUF-optimization (llama.cpp) for Ollama local-usage

In [42]:
# LOCAL SAVE: Merge LoRA adapter into base weights — required before GGUF conversion
merged_for_export = merged.merge_and_unload()
merged_for_export.save_pretrained("./qwen2.5_nano_function_master", safe_serialization=True)
tokenizer.save_pretrained("./qwen2.5_nano_function_master")
print("Full merged model saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Full merged model saved.


In [43]:
merged_for_export.push_to_hub("silvermete0r/qwen2.5-nano-function-master")
tokenizer.push_to_hub("silvermete0r/qwen2.5-nano-function-master")
print("Pushed to HuggingFace Hub.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._8y7o3r/model.safetensors:   4%|4         | 40.0MB /  988MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpuamzem_m/tokenizer.json:   0%|          | 27.6kB / 11.4MB            

Pushed to HuggingFace Hub.


In [44]:
from huggingface_hub import ModelCard

card_content = """
---
license: apache-2.0
base_model: Qwen/Qwen2.5-0.5B-Instruct
tags:
  - function-calling
  - tool-use
  - peft
  - lora
  - qwen2.5
datasets:
  - Salesforce/xlam-function-calling-60k
language:
  - en
---

# Qwen2.5-0.5B Function Calling (Fine-tuned)

A lightweight function-calling model fine-tuned from
[Qwen/Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)
on the [Salesforce/xlam-function-calling-60k](https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k) dataset.

Designed for precise, structured JSON function call generation in
resource-constrained environments.

## Training details
- **Base model**: Qwen2.5-0.5B-Instruct
- **Method**: LoRA (r=16, alpha=32)
- **Target modules**: q_proj, v_proj, k_proj, o_proj
- **Training samples**: 1500
- **Steps**: 150
- **Precision**: fp16

## Usage

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, json

model_id = "silvermete0r/qwen2.5-nano-function-master"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")

tools = [{"name": "get_weather", "description": "Get weather", "parameters": {"location": {"type": "string"}}}]

messages = [
    {"role": "system", "content": f"You have access to:\\n{json.dumps(tools)}\\nRespond strictly with a JSON array of function calls."},
    {"role": "user",   "content": "What's the weather in Paris?"},
]

inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
outputs = model.generate(inputs, max_new_tokens=128, do_sample=False)
print(tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True))
```

## Benchmark results (100-sample (random) test split)

```
Metric                        Before      After      Delta
---------------------------------------------------------
json_valid %                    41.0       96.0      +55.0
name_match %                    24.0       94.0      +70.0
args_keys_match %               14.0       79.0      +65.0
args_exact %                    10.0       69.0      +59.0
```
"""

card = ModelCard(card_content)
card.push_to_hub("silvermete0r/qwen2.5-nano-function-master")
print("Model card pushed.")

Model card pushed.
